# 01 — Exploratory Data Analysis (EDA)

Notebook ini digunakan untuk memahami **`aligned_daily.csv`** sebelum eksperimen machine learning.

Analisis yang dilakukan:
1. Struktur dan kualitas data
2. Distribusi target `direction`
3. Pergerakan JISDOR dan `log_return`
4. Distribusi jumlah berita (`n_news`)
5. Distribusi `mean_relevance`
6. Hubungan fitur berita dengan arah kurs
7. Missing value
8. Pemeriksaan tanggal
9. **Temporal split** menjadi Train–Validation–Test

> **Catatan:** karena data merupakan time-series, split dilakukan berdasarkan urutan waktu, bukan random. Random split dapat menyebabkan data dari masa depan masuk ke training dan membuat evaluasi terlalu optimistis (temporal leakage).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Jalankan notebook dari root repository.
DATA_PATH = Path("../data/processed/aligned_daily.csv")

if not DATA_PATH.exists():
    # Fallback jika notebook dijalankan dari root repository.
    DATA_PATH = Path("data/processed/aligned_daily.csv")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
df.head()

ModuleNotFoundError: No module named 'numpy'

## 1. Struktur dan kualitas data

In [ ]:
print("Kolom:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nRingkasan numerik:")
display(df.describe(include="all").T)

## 2. Distribusi target `direction`

In [ ]:
direction_counts = df["direction"].value_counts(dropna=False)

display(direction_counts.to_frame("count"))
display((direction_counts / len(df) * 100).round(2).to_frame("percentage"))

ax = direction_counts.plot(kind="bar")
ax.set_title("Distribusi Direction")
ax.set_xlabel("Direction")
ax.set_ylabel("Jumlah Hari")
plt.xticks(rotation=0)
plt.show()

## 3. Pergerakan JISDOR dan `log_return`

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["date"], df["kurs"])
ax.set_title("Pergerakan JISDOR")
ax.set_xlabel("Tanggal")
ax.set_ylabel("JISDOR (USD/IDR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
log_return = df["log_return"].dropna()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.loc[df["log_return"].notna(), "date"], log_return)
ax.axhline(0, linewidth=1)
ax.set_title("Log Return JISDOR")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Log Return")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(df["log_return"].describe())

## 4. Distribusi jumlah berita (`n_news`)

In [ ]:
display(df["n_news"].describe().to_frame("n_news"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["n_news"].dropna(), bins=30)
ax.set_title("Distribusi Jumlah Berita per Hari")
ax.set_xlabel("Jumlah Berita")
ax.set_ylabel("Jumlah Hari")
plt.tight_layout()
plt.show()

print("Hari tanpa berita:", (df["n_news"] == 0).sum())
print("Hari dengan berita:", (df["n_news"] > 0).sum())

## 5. Distribusi `mean_relevance`

In [ ]:
display(df["mean_relevance"].describe().to_frame("mean_relevance"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["mean_relevance"].dropna(), bins=20)
ax.set_title("Distribusi Mean Relevance")
ax.set_xlabel("Mean Relevance")
ax.set_ylabel("Jumlah Hari")
plt.tight_layout()
plt.show()

## 6. Hubungan fitur berita dengan arah kurs

In [ ]:
# Ringkasan fitur berita berdasarkan direction
news_by_direction = (
    df.groupby("direction", dropna=False)[["n_news", "mean_relevance", "max_relevance", "mean_tokens", "pct_offhours"]]
      .agg(["count", "mean", "median"])
)

display(news_by_direction)

In [ ]:
# Rata-rata jumlah berita per direction
mean_news = df.groupby("direction")["n_news"].mean().sort_values()

ax = mean_news.plot(kind="bar")
ax.set_title("Rata-rata Jumlah Berita berdasarkan Direction")
ax.set_xlabel("Direction")
ax.set_ylabel("Rata-rata n_news")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Boxplot jumlah berita menurut direction
groups = []
labels = []

for label, group in df.groupby("direction"):
    groups.append(group["n_news"].dropna())
    labels.append(label)

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(groups, labels=labels)
ax.set_title("Distribusi n_news berdasarkan Direction")
ax.set_xlabel("Direction")
ax.set_ylabel("Jumlah Berita")
plt.tight_layout()
plt.show()

### Catatan interpretasi

Bagian ini hanya mengeksplorasi **asosiasi/pola awal**. Perbedaan jumlah berita atau sentiment/relevance antar-kelas **belum membuktikan hubungan kausal** dan belum berarti fitur tersebut akan meningkatkan performa model.

## 7. Missing value

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

display(missing.sort_values("missing_count", ascending=False))

## 8. Pemeriksaan tanggal dan potensi masalah temporal

In [ ]:
# Cek duplikasi tanggal
print("Jumlah tanggal duplikat:", df["date"].duplicated().sum())

# Cek tanggal weekend
weekend_rows = df[df["date"].dt.dayofweek >= 5][["date", "kurs", "direction"]]
print("Jumlah row weekend:", len(weekend_rows))

if len(weekend_rows) > 0:
    display(weekend_rows.head(20))

# Cek urutan waktu
print("Tanggal minimum:", df["date"].min().date())
print("Tanggal maksimum:", df["date"].max().date())
print("Urutan tanggal sudah ascending:", df["date"].is_monotonic_increasing)

> **Penting:** bila pemeriksaan di atas menemukan tanggal non-business day pada data JISDOR, jangan langsung menghapusnya. Periksa kembali proses alignment/target terlebih dahulu karena kesalahan tanggal dapat memengaruhi eksperimen secara keseluruhan.

## 9. Temporal Train–Validation–Test Split

In [ ]:
# Split kronologis 70% / 15% / 15%.
# Tidak menggunakan random split karena data memiliki urutan waktu.

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("TRAIN")
print(train["date"].min().date(), "→", train["date"].max().date(), "|", len(train), "rows")

print("\nVALIDATION")
print(validation["date"].min().date(), "→", validation["date"].max().date(), "|", len(validation), "rows")

print("\nTEST")
print(test["date"].min().date(), "→", test["date"].max().date(), "|", len(test), "rows")

In [ ]:
# Simpan split untuk dipakai notebook/model berikutnya.
SPLIT_DIR = DATA_PATH.parent / "split"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train.to_csv(SPLIT_DIR / "train.csv", index=False)
validation.to_csv(SPLIT_DIR / "validation.csv", index=False)
test.to_csv(SPLIT_DIR / "test.csv", index=False)

print("Split disimpan di:", SPLIT_DIR.resolve())

In [ ]:
# Cek distribusi target per split
split_summary = pd.DataFrame({
    "train": train["direction"].value_counts(normalize=True),
    "validation": validation["direction"].value_counts(normalize=True),
    "test": test["direction"].value_counts(normalize=True),
}).fillna(0).round(3)

display(split_summary)

## Output yang diharapkan

Setelah notebook dijalankan, kita harus punya:
- gambaran kualitas dan distribusi dataset,
- visualisasi JISDOR dan `log_return`,
- gambaran pola berita terhadap `direction`,
- daftar missing value,
- pengecekan tanggal/temporal alignment,
- `train.csv`, `validation.csv`, dan `test.csv`.

Notebook berikutnya baru digunakan untuk eksperimen model, dimulai dari **historical baseline** lalu dibandingkan dengan **TF-IDF/NLP baseline** dan **combined model**.